<a href="https://colab.research.google.com/github/Heng1222/MalSem-Decon/blob/main/concept_keyword_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Single-Concept Keyword Discovery

目的：根據 keywords 從貼文語料中擴充候選關鍵詞。

流程：
1. 載入貼文資料。
2. 合併指定文字欄位成 `text`。
3. 使用 keywords seed 做弱標註。
4. 建立 concept bucket 與 background bucket。
5. 使用 embedding 找語意相近貼文，擴大 concept bucket。
6. 使用 log-odds 找出候選關鍵詞。
7. 使用 similarity 過濾語意漂移。
8. 輸出候選詞、弱標註貼文與摘要。


In [49]:
!pip install -U pandas numpy scikit-learn sentence-transformers openpyxl

In [50]:
from __future__ import annotations

import argparse
import json
import math
import re
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

try:
    from sentence_transformers import SentenceTransformer
except ImportError:  # 讓非 embedding 模式仍可執行
    SentenceTransformer = None

## Config

In [51]:
@dataclass
class DiscoveryConfig:
    concept_name: str = "investment_scam"
    input_path: str = "time_line.xlsx"
    output_dir: str = "output_single_concept"

    # 貼文欄位。若之後要改成社團名稱，只要改成 ["name"] 或你的社團名稱欄位即可。
    text_columns: list[str] = field(default_factory=lambda: [
        "content",
        "share_content",
        "attachment_title",
        "attachment_description",
    ])

    # 單一 concept 的 seed；不要混不同 concept。
    seed_keywords: list[str] = field(default_factory=lambda: [
        "詐騙", "假投資", "投資詐騙", "帶單", "保證獲利", "穩賺", "飆股", "明牌", "代操", "入金", "出金"
    ])

    # 排除「談論詐騙」但不是詐騙操作語意的脈絡，可依資料調整。先為空
    exclude_keywords: list[str] = field(default_factory=lambda: [])

    # 一般內容 seed 只用來建立較乾淨的 background；若資料不足，程式會退回 non-concept background。
    benign_keywords: list[str] = field(default_factory=lambda: [
        "美食", "料理", "運動", "健身", "旅遊", "寵物", "攝影", "音樂", "電影", "閱讀", "親子", "露營", "烘焙", "手作"
    ])

    stop_terms: set[str] = field(default_factory=lambda: {
        "今天", "分享", "可以", "一起", "自己", "這個", "如果", "因為", "就是", "不是",
        "一個", "你們", "我們", "他們", "真的", "看到", "知道", "感謝", "希望", "喜歡",
        "支持", "朋友", "留言", "歡迎", "最新", "大家", "需要", "使用", "生活", "影片",
        "照片", "粉絲", "覺得", "是否", "開始", "之後", "直接", "相關", "提供", "目前",
        "收到", "免費", "時間", "內容", "文章", "連結", "作者", "社團", "社群",
    })

    # n-gram 設定
    min_cjk_ngram: int = 2
    max_cjk_ngram: int = 4
    min_token_len: int = 2

    # 抽樣與候選設定
    max_concept_docs: int = 500
    max_background_docs: int = 3000
    top_k_candidates: int = 300
    min_candidate_freq: int = 2
    smoothing: float = 0.1
    random_state: int = 42

    # embedding 語意擴張設定
    use_embedding_expansion: bool = True
    model_name: str = "paraphrase-multilingual-MiniLM-L12-v2"
    top_similar_docs: int = 300
    similarity_threshold: float = 0.45
    candidate_similarity_threshold: float = 0.42

    # 是否把 seed 也放進輸出，方便後續人工整理
    include_seeds_in_output: bool = True

## IO

In [52]:
CJK_OR_ALNUM = re.compile(r"[\u4e00-\u9fff]+|[a-z0-9_]+")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WHITESPACE_RE = re.compile(r"\s+")

def read_xlsx_without_openpyxl(path: str | Path) -> pd.DataFrame:
    """直接解析 xlsx 的 XML。若環境沒有 openpyxl，可用這個 fallback。"""
    path = Path(path)
    ns = {"a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

    def col_name(cell_ref: str) -> str:
        return "".join(ch for ch in cell_ref if ch.isalpha())

    def cell_text(cell, shared_strings: list[str]) -> str:
        cell_type = cell.attrib.get("t")
        inline = cell.find("a:is", ns)
        value = cell.find("a:v", ns)
        if inline is not None:
            return "".join(inline.itertext())
        if value is None or value.text is None:
            return ""
        if cell_type == "s":
            idx = int(value.text)
            return shared_strings[idx] if idx < len(shared_strings) else ""
        return value.text

    with zipfile.ZipFile(path) as zf:
        shared_strings: list[str] = []
        if "xl/sharedStrings.xml" in zf.namelist():
            ss_root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in ss_root.findall("a:si", ns):
                shared_strings.append("".join(si.itertext()))

        sheet_names = [name for name in zf.namelist() if name.startswith("xl/worksheets/sheet") and name.endswith(".xml")]
        if not sheet_names:
            raise ValueError(f"找不到 worksheet：{path}")
        root = ET.fromstring(zf.read(sheet_names[0]))
        sheet_rows = root.find("a:sheetData", ns)
        if sheet_rows is None:
            return pd.DataFrame()

        rows: list[dict[str, str]] = []
        header_map: dict[str, str] = {}
        for row_idx, row in enumerate(sheet_rows):
            row_values: dict[str, str] = {}
            for cell in row:
                ref = cell.attrib.get("r", "")
                col = col_name(ref)
                row_values[col] = cell_text(cell, shared_strings)
            if row_idx == 0:
                header_map = row_values
            else:
                rows.append({header_map.get(col, col): value for col, value in row_values.items()})
    return pd.DataFrame(rows)


def read_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt"}:
        return pd.read_csv(path, encoding="utf-8")
    if suffix in {".xlsx", ".xlsm", ".xls"}:
        try:
            return pd.read_excel(path)
        except Exception:
            return read_xlsx_without_openpyxl(path)
    raise ValueError(f"不支援的檔案格式：{suffix}")


def load_keywords(path_or_values: Optional[str | Iterable[str]]) -> list[str]:
    if path_or_values is None:
        return []
    if isinstance(path_or_values, str):
        p = Path(path_or_values)
        if p.exists():
            if p.suffix.lower() == ".json":
                data = json.loads(p.read_text(encoding="utf-8"))
                if isinstance(data, dict):
                    # 若是 {concept: [keywords]}，取第一組；CLI 建議直接用 --seeds_json 指定完整 config。
                    first_value = next(iter(data.values()))
                    return [str(x).strip() for x in first_value if str(x).strip()]
                return [str(x).strip() for x in data if str(x).strip()]
            return (
                pd.read_csv(p, header=None, encoding="utf-8")[0]
                .dropna()
                .astype(str)
                .str.strip()
                .loc[lambda s: s != ""]
                .tolist()
            )
        return [x.strip() for x in path_or_values.split(",") if x.strip()]
    return [str(x).strip() for x in path_or_values if str(x).strip()]

## Text processing

In [53]:
def normalize_text(text) -> str:
    text = "" if text is None else str(text)
    text = text.replace("\ufeff", " ")
    text = URL_RE.sub(" ", text.lower())
    text = re.sub(r"[\r\n\t]+", " ", text)
    return WHITESPACE_RE.sub(" ", text).strip()


def tokenize_mixed(text: str, cfg: DiscoveryConfig) -> list[str]:
    text = normalize_text(text)
    tokens: list[str] = []
    for block in CJK_OR_ALNUM.findall(text):
        if re.fullmatch(r"[a-z0-9_]+", block):
            if len(block) >= cfg.min_token_len and block not in cfg.stop_terms:
                tokens.append(block)
            continue
        if len(block) < cfg.min_cjk_ngram:
            continue
        upper = min(cfg.max_cjk_ngram, len(block))
        for n in range(cfg.min_cjk_ngram, upper + 1):
            for i in range(len(block) - n + 1):
                term = block[i : i + n]
                if term not in cfg.stop_terms:
                    tokens.append(term)
    return tokens


def keyword_hits(text: str, keywords: list[str]) -> list[str]:
    text = normalize_text(text)
    # 長詞優先，輸出比較可讀。
    return [kw for kw in sorted(set(keywords), key=len, reverse=True) if kw and kw in text]


def build_text_column(df: pd.DataFrame, text_columns: list[str]) -> pd.Series:
    for col in text_columns:
        if col not in df.columns:
            df[col] = ""
    return df[text_columns].fillna("").astype(str).agg(" ".join, axis=1).map(normalize_text)

## Statistics

In [54]:
def get_word_frequencies(texts: Iterable[str], cfg: DiscoveryConfig) -> Counter:
    counts: Counter = Counter()
    for text in texts:
        counts.update(tokenize_mixed(text, cfg))
    return counts


def compute_log_odds(target_counts: Counter, ref_counts: Counter, cfg: DiscoveryConfig) -> pd.DataFrame:
    terms = sorted(set(target_counts) | set(ref_counts))
    n_target = sum(target_counts.values())
    n_ref = sum(ref_counts.values())
    rows = []
    alpha = cfg.smoothing

    for term in terms:
        if target_counts[term] < cfg.min_candidate_freq:
            continue
        p = (target_counts[term] + alpha) / (n_target + alpha * max(1, len(terms)))
        q = (ref_counts[term] + alpha) / (n_ref + alpha * max(1, len(terms)))
        p = min(max(p, 1e-12), 1 - 1e-12)
        q = min(max(q, 1e-12), 1 - 1e-12)
        log_odds = math.log(p / (1 - p)) - math.log(q / (1 - q))
        rows.append({
            "term": term,
            "log_odds": log_odds,
            "concept_freq": int(target_counts[term]),
            "background_freq": int(ref_counts[term]),
        })

    return pd.DataFrame(rows).sort_values("log_odds", ascending=False).reset_index(drop=True)

## Embedding helpers

In [55]:
def load_model(cfg: DiscoveryConfig):
    if not cfg.use_embedding_expansion:
        return None
    if SentenceTransformer is None:
        raise ImportError("尚未安裝 sentence-transformers。請執行：pip install -U sentence-transformers")
    print(f"載入 embedding model：{cfg.model_name}")
    return SentenceTransformer(cfg.model_name)


def encode_texts(model, texts: list[str]) -> np.ndarray:
    return model.encode(texts, show_progress_bar=True, normalize_embeddings=True)


def semantic_expand_bucket(
    posts: pd.DataFrame,
    concept_mask: pd.Series,
    background_mask: pd.Series,
    cfg: DiscoveryConfig,
    model,
) -> tuple[pd.DataFrame, pd.DataFrame, Optional[np.ndarray]]:
    """用 seed 命中的 concept docs 建 center，再找語意相近貼文。"""
    concept_seed_docs = posts.loc[concept_mask].copy()
    background_docs = posts.loc[background_mask].copy()

    if concept_seed_docs.empty:
        raise ValueError("concept bucket 為空。請增加或修正 seed_keywords。")

    if model is None:
        concept_docs = concept_seed_docs
        return concept_docs, background_docs, None

    all_texts = posts["text"].tolist()
    all_embeddings = encode_texts(model, all_texts)
    seed_idx = np.flatnonzero(concept_mask.to_numpy())
    concept_center = all_embeddings[seed_idx].mean(axis=0)
    concept_center = concept_center / max(np.linalg.norm(concept_center), 1e-12)

    sims = cosine_similarity(concept_center.reshape(1, -1), all_embeddings)[0]
    posts = posts.copy()
    posts["concept_similarity"] = sims

    top_n = min(cfg.top_similar_docs, len(posts))
    top_idx = np.argsort(sims)[-top_n:][::-1]
    sim_mask = pd.Series(False, index=posts.index)
    sim_mask.iloc[top_idx] = True
    sim_mask = sim_mask | concept_mask

    # 排除反向脈絡或明顯 exclude 的內容。
    expanded = posts.loc[sim_mask & ~posts["has_exclude_context"]].copy()
    if len(expanded) > cfg.max_concept_docs:
        expanded = expanded.sort_values("concept_similarity", ascending=False).head(cfg.max_concept_docs)

    # background 盡量取低相似、未命中 concept、未命中 exclude 的內容。
    bg = posts.loc[~concept_mask & ~posts["has_exclude_context"]].copy()
    bg = bg.sort_values("concept_similarity", ascending=True)
    if len(bg) > cfg.max_background_docs:
        bg = bg.head(cfg.max_background_docs)

    return expanded, bg, concept_center


def add_candidate_similarity(candidates: pd.DataFrame, cfg: DiscoveryConfig, model, concept_center: Optional[np.ndarray]) -> pd.DataFrame:
    candidates = candidates.copy()
    if model is None or concept_center is None or candidates.empty:
        candidates["similarity"] = np.nan
        return candidates
    term_embeddings = model.encode(candidates["term"].tolist(), show_progress_bar=False, normalize_embeddings=True)
    sims = cosine_similarity(term_embeddings, concept_center.reshape(1, -1))[:, 0]
    candidates["similarity"] = sims
    return candidates

## Main pipeline

In [56]:
def discover_single_concept_keywords(cfg: DiscoveryConfig) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    np.random.seed(cfg.random_state)
    out_dir = Path(cfg.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    df = read_table(cfg.input_path)
    df = df.copy()
    df["text"] = build_text_column(df, cfg.text_columns)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)

    if df.empty:
        raise ValueError("輸入資料沒有可用文字。請確認 text_columns 設定。")

    # 弱標註：單一 concept 的 seed 命中。
    df["concept_hits"] = df["text"].map(lambda t: keyword_hits(t, cfg.seed_keywords))
    df["concept_seed_score"] = df["concept_hits"].str.len()
    df["exclude_hits"] = df["text"].map(lambda t: keyword_hits(t, cfg.exclude_keywords))
    df["has_exclude_context"] = df["exclude_hits"].str.len() > 0
    df["benign_hits"] = df["text"].map(lambda t: keyword_hits(t, cfg.benign_keywords))
    df["benign_seed_score"] = df["benign_hits"].str.len()

    concept_mask = (df["concept_seed_score"] > 0) & (~df["has_exclude_context"])
    benign_mask = (df["concept_seed_score"] == 0) & (df["benign_seed_score"] > 0) & (~df["has_exclude_context"])
    fallback_background_mask = (df["concept_seed_score"] == 0) & (~df["has_exclude_context"])

    # 若一般 seed 背景太少，退回「未命中 concept」背景。
    background_mask = benign_mask if benign_mask.sum() >= 30 else fallback_background_mask

    model = load_model(cfg)
    concept_docs, background_docs, concept_center = semantic_expand_bucket(
        posts=df,
        concept_mask=concept_mask,
        background_mask=background_mask,
        cfg=cfg,
        model=model,
    )

    # 若無 embedding，這裡控制背景抽樣大小。
    if len(background_docs) > cfg.max_background_docs:
        background_docs = background_docs.sample(cfg.max_background_docs, random_state=cfg.random_state)

    concept_counts = get_word_frequencies(concept_docs["text"], cfg)
    background_counts = get_word_frequencies(background_docs["text"], cfg)
    candidates = compute_log_odds(concept_counts, background_counts, cfg)

    # 排除原 seed、exclude、stop terms，保留真正 candidate；seed 另附在輸出上方。
    seed_set = set(cfg.seed_keywords)
    exclude_set = set(cfg.exclude_keywords)
    candidates = candidates.loc[
        ~candidates["term"].isin(seed_set)
        & ~candidates["term"].isin(exclude_set)
        & ~candidates["term"].isin(cfg.stop_terms)
    ].copy()

    candidates = add_candidate_similarity(candidates, cfg, model, concept_center)

    if cfg.use_embedding_expansion and "similarity" in candidates.columns:
        candidates["decision"] = np.select(
            [
                candidates["similarity"].ge(cfg.candidate_similarity_threshold) & candidates["log_odds"].gt(0),
                candidates["log_odds"].gt(0),
            ],
            ["A_candidate", "B_context_candidate"],
            default="C_low_confidence",
        )
        candidates = candidates.sort_values(["decision", "log_odds", "similarity"], ascending=[True, False, False])
    else:
        candidates["decision"] = np.where(candidates["log_odds"].gt(0), "B_context_candidate", "C_low_confidence")
        candidates = candidates.sort_values("log_odds", ascending=False)

    candidates.insert(0, "concept", cfg.concept_name)
    candidates = candidates.head(cfg.top_k_candidates).reset_index(drop=True)

    if cfg.include_seeds_in_output:
        seed_rows = pd.DataFrame({
            "concept": cfg.concept_name,
            "term": cfg.seed_keywords,
            "log_odds": np.nan,
            "concept_freq": [int(concept_counts.get(s, 0)) for s in cfg.seed_keywords],
            "background_freq": [int(background_counts.get(s, 0)) for s in cfg.seed_keywords],
            "similarity": np.nan,
            "decision": "seed",
        })
        candidates = pd.concat([seed_rows, candidates], ignore_index=True)

    summary = {
        "concept_name": cfg.concept_name,
        "total_docs": int(len(df)),
        "seed_concept_docs": int(concept_mask.sum()),
        "expanded_concept_docs": int(len(concept_docs)),
        "background_docs": int(len(background_docs)),
        "candidate_terms": int((candidates["decision"] != "seed").sum()),
        "used_embedding_expansion": bool(cfg.use_embedding_expansion),
    }

    df.to_csv(out_dir / f"{cfg.concept_name}_labeled_posts.csv", index=False, encoding="utf-8-sig")
    candidates.to_csv(out_dir / f"{cfg.concept_name}_keyword_candidates.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame([summary]).to_csv(out_dir / f"{cfg.concept_name}_summary.csv", index=False, encoding="utf-8-sig")
    (out_dir / f"{cfg.concept_name}_config.json").write_text(
        json.dumps({k: (list(v) if isinstance(v, set) else v) for k, v in cfg.__dict__.items()}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("完成。輸出：")
    print(f"- {out_dir / f'{cfg.concept_name}_keyword_candidates.csv'}")
    print(f"- {out_dir / f'{cfg.concept_name}_labeled_posts.csv'}")
    print(f"- {out_dir / f'{cfg.concept_name}_summary.csv'}")
    print("摘要：", summary)

    return candidates, df, summary

## 設定本次要跑的 concept

請先確認 `input_path` 與 `text_columns`。

- 貼文資料：`text_columns` 可使用 `content, share_content, attachment_title, attachment_description`。
- 社團名稱資料：之後只要改成社團名稱欄位，例如 `text_columns = ["name"]`，其餘流程可以共用。

In [57]:
cfg = DiscoveryConfig(
    concept_name="topic",
    input_path="time_line.xlsx",
    output_dir="./topic",
    text_columns=[
        "content",
        "share_content",
        "attachment_title",
        "attachment_description",
    ],
    # seed_keywords=['八大', '夜場', '酒店', '招待所'],
    seed_keywords=['貸款', '信貸', '快易貸', '企業貸', '債務', '破產', '法拍', '卡債', '八大', '精品', '娛樂城', '炫富', '厭世', '致富', '借款', '二胎', '自救會', '清算', '日領', '不問經歷', '夜場', '酒店', '全額', '強力過件', '協商', '免頭期', '槓桿', '運彩', '招待所', '博弈'],
    exclude_keywords=[],
    use_embedding_expansion=True,
    top_similar_docs=200,
    top_k_candidates=300,
    candidate_similarity_threshold=0.42,
)

cfg

DiscoveryConfig(concept_name='topic', input_path='time_line.xlsx', output_dir='./topic', text_columns=['content', 'share_content', 'attachment_title', 'attachment_description'], seed_keywords=['貸款', '信貸', '快易貸', '企業貸', '債務', '破產', '法拍', '卡債', '八大', '精品', '娛樂城', '炫富', '厭世', '致富', '借款', '二胎', '自救會', '清算', '日領', '不問經歷', '夜場', '酒店', '全額', '強力過件', '協商', '免頭期', '槓桿', '運彩', '招待所', '博弈'], exclude_keywords=[], benign_keywords=['美食', '料理', '運動', '健身', '旅遊', '寵物', '攝影', '音樂', '電影', '閱讀', '親子', '露營', '烘焙', '手作'], stop_terms={'因為', '是否', '支持', '相關', '可以', '之後', '連結', '真的', '喜歡', '歡迎', '知道', '時間', '最新', '我們', '感謝', '生活', '作者', '社團', '你們', '收到', '就是', '這個', '今天', '大家', '如果', '看到', '直接', '他們', '免費', '需要', '使用', '提供', '分享', '不是', '目前', '留言', '文章', '社群', '一個', '照片', '開始', '粉絲', '內容', '希望', '覺得', '一起', '自己', '影片', '朋友'}, min_cjk_ngram=2, max_cjk_ngram=4, min_token_len=2, max_concept_docs=500, max_background_docs=3000, top_k_candidates=300, min_candidate_freq=2, smoothing=0.1, random_state=42, use_embeddi

## 執行 keyword discovery

In [58]:
candidates, labeled_posts, summary = discover_single_concept_keywords(cfg)
summary

載入 embedding model：paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1190 [00:00<?, ?it/s]

完成。輸出：
- topic/topic_keyword_candidates.csv
- topic/topic_labeled_posts.csv
- topic/topic_summary.csv
摘要： {'concept_name': 'topic', 'total_docs': 38069, 'seed_concept_docs': 329, 'expanded_concept_docs': 500, 'background_docs': 3000, 'candidate_terms': 300, 'used_embedding_expansion': True}


{'concept_name': 'topic',
 'total_docs': 38069,
 'seed_concept_docs': 329,
 'expanded_concept_docs': 500,
 'background_docs': 3000,
 'candidate_terms': 300,
 'used_embedding_expansion': True}

## 檢視候選關鍵詞

In [59]:
for i in range(5):
  print(candidates[(i-1)*50:i*50][["term", "log_odds", "concept_freq", "background_freq", "similarity"]])

Empty DataFrame
Columns: [term, log_odds, concept_freq, background_freq, similarity]
Index: []
    term  log_odds  concept_freq  background_freq  similarity
0     貸款       NaN           225                0         NaN
1     信貸       NaN             5                0         NaN
2    快易貸       NaN             0                0         NaN
3    企業貸       NaN             0                0         NaN
4     債務       NaN             8                0         NaN
5     破產       NaN             8                0         NaN
6     法拍       NaN             2                0         NaN
7     卡債       NaN             1                0         NaN
8     八大       NaN            41                0         NaN
9     精品       NaN            77                0         NaN
10   娛樂城       NaN            10                0         NaN
11    炫富       NaN             1                0         NaN
12    厭世       NaN            22                0         NaN
13    致富       NaN            13     

## 檢視弱標註貼文樣本

In [60]:
labeled_posts[["text", "concept_hits", "exclude_hits", "benign_hits"]].head(20)

,text,concept_hits,exclude_hits,benign_hits
0,有一個雞蛋，非常天真的和石頭在一起了 磕磕碰碰，弄的自己身上傷痕累累 。 但雞蛋一直堅持忍耐...,[],[],[]
1,有多少次 你把那些傷痛說得雲淡風輕 用開玩笑的方式去化解尷尬 只為了不想解釋 ... 那些只...,[],[],[]
2,刀子嘴豆腐心❤️,[],[],[]
3,你離開後，失落的處女會忘記，怎麼再愛上別人 - 小星星 你離開後，失落的處女會忘記，怎麼再愛上別人,[],[],[]
4,但願如此,[],[],[]
5,人言可畏,[],[],[]
6,不改己錯，難得幸福 一位年輕人和新婚的妻子經常吵架， 他很懊悔，埋怨自己當初結婚前看走了眼....,[],[],[]
7,“我还爱你，但是我不喜欢你了。” 我想这句话的意思是， 你仍然美好的让我心动， 可我已经没有...,[],[],[]
8,什麼是舒服的感情 就是不把自己的問題讓對方困擾 每個人都管理好自己 就不會用情緒去要求對方來...,[],[],[]
9,「 熬過這些 我們就結婚吧 」 我不需要你的頭像是我 也不需要你朋友圈背景是我 也不需要弄的...,[],[],[]
